In [1]:
import numpy as np
from aeon.transformations.collection.convolution_based import MultiRocket
from aeon.transformations.collection.convolution_based._hydra import HydraTransformer
from aeon.utils.validation import check_n_jobs
from aeon.transformations.collection.interval_based import QUANTTransformer
import numpy as np
import polars as pl
from aeon.classification.base import BaseClassifier
from aeon.classification.feature_based import (
    Catch22Classifier,
)
import os
from aeon.transformations.collection.convolution_based import Rocket
from aeon.datasets.tsc_datasets import univariate
from sklearn.base import clone
from aeon.classification.convolution_based import MultiRocketHydraClassifier
from aeon.classification.convolution_based import RocketClassifier
from sklearn.metrics import accuracy_score
from aeon.classification.interval_based import QUANTClassifier
from autotsc import utils, models, transformers
from tqdm import tqdm
from aeon.classification.feature_based import Catch22Classifier
from aeon.classification.interval_based import QUANTClassifier
from aeon.classification.shapelet_based import RDSTClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifierCV
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from aeon.pipeline import make_pipeline as aeon_make_pipeline
from aeon.transformations.collection import Normalizer
import seaborn as sns
import matplotlib.pyplot as plt

2025-12-09 19:45:17.198467: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def get_model(transform, random_state):
    if transform == 'downsample':
        return aeon_make_pipeline(
            transformers.DownsampleTransformer(proportion=0.5),
            MultiRocketHydraClassifier(n_jobs=-1, random_state=random_state)
        )
    elif transform == 'difference':
        return aeon_make_pipeline(
            transformers.Difference(),
            MultiRocketHydraClassifier(n_jobs=-1, random_state=random_state)
        )
    elif transform == 'cumsum':
        return aeon_make_pipeline(
            transformers.CumSum(),
            MultiRocketHydraClassifier(n_jobs=-1, random_state=random_state)
        )
    elif transform == 'scale':
        return aeon_make_pipeline(
            Normalizer(),
            MultiRocketHydraClassifier(n_jobs=-1, random_state=random_state)
        )
    elif transform == 'polar-angle':
        return aeon_make_pipeline(
            transformers.PolarCoordinates(mode='angle'),
            MultiRocketHydraClassifier(n_jobs=-1, random_state=random_state)
        )
    elif transform == 'polar-magnitude':
        return aeon_make_pipeline(
            transformers.PolarCoordinates(mode='magnitude'),
            MultiRocketHydraClassifier(n_jobs=-1, random_state=random_state)
        )
    elif transform == 'rank':
        return aeon_make_pipeline(
            transformers.RankTransform(),
            MultiRocketHydraClassifier(n_jobs=-1, random_state=random_state)
        )
    elif transform == 'none':
        return MultiRocketHydraClassifier(n_jobs=-1, random_state=random_state)
    else:
        raise ValueError(f"Unknown transform: {transform}")



In [3]:
write_dir = "experiments/automl_series_transformations"
os.makedirs(write_dir, exist_ok=True)

In [ ]:
for dataset in univariate:
    for run in range(1):
        for model_name in ['none', 'downsample', 'difference', 'cumsum', 'scale', 'polar-angle', 'polar-magnitude', 'rank']:
            print(f"{dataset} {run} {model_name}")
            stats = {
                'dataset': dataset,
                'run': run,
                'model': model_name,
            }

            hash_val = pl.DataFrame([stats]).hash_rows(seed=42, seed_1=1, seed_2=2, seed_3=3).item()
            file = f"{write_dir}/{hash_val}.parquet"

            if os.path.exists(file):
                print(f"Skipping: Dataset={dataset}, Run={run}, Model={model_name}")
                continue
            else:
                print(f"Processing: Dataset={dataset}, Run={run}, Model={model_name}")


            model = get_model(model_name, random_state=run)
            X_train, y_train, X_test, y_test = utils.load_dataset(dataset)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test, y_pred)

            stats['accuracy'] = acc

            df_stat = pl.DataFrame([stats])
            df_stat.write_parquet(file, mkdir=True)

ACSF1 0 none
Skipping: Dataset=ACSF1, Run=0, Model=none
ACSF1 0 downsample
Skipping: Dataset=ACSF1, Run=0, Model=downsample
ACSF1 0 difference
Skipping: Dataset=ACSF1, Run=0, Model=difference
ACSF1 0 cumsum
Skipping: Dataset=ACSF1, Run=0, Model=cumsum
ACSF1 0 scale
Skipping: Dataset=ACSF1, Run=0, Model=scale
ACSF1 0 polar-angle
Skipping: Dataset=ACSF1, Run=0, Model=polar-angle
ACSF1 0 polar-magnitude
Skipping: Dataset=ACSF1, Run=0, Model=polar-magnitude
ACSF1 0 rank
Skipping: Dataset=ACSF1, Run=0, Model=rank
Adiac 0 none
Skipping: Dataset=Adiac, Run=0, Model=none
Adiac 0 downsample
Skipping: Dataset=Adiac, Run=0, Model=downsample
Adiac 0 difference
Skipping: Dataset=Adiac, Run=0, Model=difference
Adiac 0 cumsum
Skipping: Dataset=Adiac, Run=0, Model=cumsum
Adiac 0 scale
Skipping: Dataset=Adiac, Run=0, Model=scale
Adiac 0 polar-angle
Skipping: Dataset=Adiac, Run=0, Model=polar-angle
Adiac 0 polar-magnitude
Skipping: Dataset=Adiac, Run=0, Model=polar-magnitude
Adiac 0 rank
Skipping: Datas

In [ ]:
df = pl.read_parquet(write_dir + "/*.parquet")
df

In [ ]:
pdf = df.pivot(values="accuracy", index="dataset", columns="model", aggregate_function='mean').drop_nulls()
models = df["model"].unique().to_list()
pdf

In [ ]:
from aeon.visualisation import plot_critical_difference
plot_critical_difference(pdf.select(models).to_numpy(), models)